In [41]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import xgboost as xgb

data = pd.read_csv("dataset/final_coastal_dataset.csv")

data["Date"] = pd.to_datetime(data["Date"])
data["Rainfall_lag2"] = data["Rainfall"].shift(2)
data["Rainfall_lag3"] = data["Rainfall"].shift(3)

data = data.dropna()

print(data.head())


        Date    Rainfall  Sea_Level  Soil_Moisture  Rainfall_lag1  \
3 1998-06-30   28.041772     7055.0       3.988700      21.676685   
4 1998-07-31   84.962687     7055.0       4.983455      28.041772   
5 1998-08-31  134.570285     7055.0       5.575614      84.962687   
6 1998-09-30  133.236587     7055.0       5.476501     134.570285   
7 1998-10-31  215.872284     7055.0       5.725842     133.236587   

   Rainfall_3month_avg  Sea_Level_trend  Rainfall_lag2  Rainfall_lag3  
3            21.919706              0.0      16.040662       1.248927  
4            44.893715              0.0      21.676685      16.040662  
5            82.524915              0.0      28.041772      21.676685  
6           117.589853              0.0      84.962687      28.041772  
7           161.226385              0.0     134.570285      84.962687  


In [42]:
X = data[[
    "Rainfall",
    "Soil_Moisture",
    "Rainfall_lag1",
    "Rainfall_lag2",
    "Rainfall_lag3",
    "Rainfall_3month_avg",
    "Sea_Level_trend"
]]

y = data["Sea_Level"]
split_index = int(len(data) * 0.8)

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]
print(X_train.shape, X_test.shape)

(159, 7) (40, 7)


In [43]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [44]:
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest MAE:", rf_mae)
print("Random Forest R2:", rf_r2)

Random Forest MAE: 44.83943749999994
Random Forest R2: 0.5905321953175358


In [45]:
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [46]:
xgb_pred = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost MAE:", xgb_mae)
print("XGBoost R2:", xgb_r2)

XGBoost MAE: 48.08673095703125
XGBoost R2: 0.4910988253970169


In [47]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3
)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)

from sklearn.metrics import mean_absolute_error, r2_score

gb_mae = mean_absolute_error(y_test, gb_pred)
gb_r2 = r2_score(y_test, gb_pred)

print("Gradient Boosting MAE:", gb_mae)
print("Gradient Boosting R2:", gb_r2)

Gradient Boosting MAE: 42.92269256400827
Gradient Boosting R2: 0.5826481060271855


In [48]:
ensemble_pred = (rf_pred + xgb_pred + gb_pred) / 3

ensemble_mae = mean_absolute_error(y_test, ensemble_pred)
ensemble_r2 = r2_score(y_test, ensemble_pred)

print("Ensemble MAE:", ensemble_mae)
print("Ensemble R2:", ensemble_r2)

Ensemble MAE: 43.431269678342005
Ensemble R2: 0.5863564727146933


In [49]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression

stack_model = StackingRegressor(
    estimators=[
        ("rf", rf_model),
        ("xgb", xgb_model),
        ("gb", gb_model)
    ],
    final_estimator=LinearRegression()
)

stack_model.fit(X_train, y_train)

stack_pred = stack_model.predict(X_test)

stack_mae = mean_absolute_error(y_test, stack_pred)
stack_r2 = r2_score(y_test, stack_pred)

print("Stacking MAE:", stack_mae)
print("Stacking R2:", stack_r2)

Stacking MAE: 42.71322758865758
Stacking R2: 0.6179047996280156


In [59]:
def classify_risk(sea_level):

    if sea_level < 7000:
        return "Low"

    elif 7000 <= sea_level <= 7200:
        return "Medium"

    else:
        return "High"

data["Flood_Risk"] = data["Sea_Level"].apply(classify_risk)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data["Flood_Risk_Label"] = le.fit_transform(data["Flood_Risk"])

print(data[["Sea_Level", "Flood_Risk"]].head())

   Sea_Level Flood_Risk
3     7055.0     Medium
4     7055.0     Medium
5     7055.0     Medium
6     7055.0     Medium
7     7055.0     Medium


In [65]:
data["Month"] = data["Date"].dt.month
X = data[[
    "Rainfall",
    "Soil_Moisture",
    "Rainfall_lag1",
    "Rainfall_lag2",
    "Rainfall_lag3",
    "Rainfall_3month_avg",
    "Sea_Level_trend",
    "Month"
]]

# y = data["Flood_Risk"]
y = data["Flood_Risk_Label"]

In [66]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [67]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_clf.fit(X_train, y_train)
y_pred = rf_clf.predict(X_test)

In [68]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred))

Accuracy: 0.775
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.89      0.76      0.82        21
           2       0.70      0.82      0.76        17

    accuracy                           0.78        40
   macro avg       0.70      0.70      0.69        40
weighted avg       0.79      0.78      0.78        40



In [69]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

gb_clf = GradientBoostingClassifier()

gb_clf.fit(X_train, y_train)

gb_pred = gb_clf.predict(X_test)

print("Gradient Boost Accuracy:", accuracy_score(y_test, gb_pred))

Gradient Boost Accuracy: 0.725


In [71]:
import xgboost as xgb

xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05
)

xgb_clf.fit(X_train, y_train)

xgb_pred = xgb_clf.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))

XGBoost Accuracy: 0.75


In [75]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [76]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr_clf = LogisticRegression(max_iter=3000)

lr_clf.fit(X_train_scaled, y_train)

lr_pred = lr_clf.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))

Logistic Regression Accuracy: 0.775
